In [1]:
import sys
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

sys.path.append('../reconstruction')
sys.path.append('./reconstruction')
from util import Population

from standardize_align_new import Standardizer, SingleTransformerEncoderStandardizer

from models import base_CNN, two_CNN, var_CNN, base_RNN, var_RNN, base_TransformerEncoder

generalized align


In [2]:
def print_weights(model):
    for name, param in model.named_parameters():
        if 'weight' in name:
            print(f'Layer: {name} - Weights')
            print(param)
        elif 'bias' in name:
            print(f'Layer: {name} - Biases')
            print(param)

def print_pairs(model, blackbox):
    for (name1, param1), (name2, param2) in zip(model.named_parameters(), blackbox.named_parameters()):
        if 'weight' in name1:
            print(f'Layer: {name1} - Reconstructed Weights')
            print(param1)
            print(f'Layer: {name2} - Blackbox Weights')
            print(param2)
        elif 'bias' in name1:
            print(f'Layer: {name1} - Reconstructed Biases')
            print(param1)
            print(f'Layer: {name2} - Blackbox Biases')
            print(param2)

In [3]:
#RNN

blackbox_dict_path = "rnn/seed_31_RNNx28_outer_iterations_55_num_samples_10000_num_epochs_5_dataset_mnist_optim_adam_activation_relu_sampling_method_committee_aligner_var_RNN_28/black_box.pt"
blackbox_og_params_dict_path = "rnn/seed_31_RNNx28_outer_iterations_55_num_samples_10000_num_epochs_5_dataset_mnist_optim_adam_activation_relu_sampling_method_committee_aligner_var_RNN_28/original_params_black_box.pt"
final_population_dict_path = "rnn/seed_31_RNNx28_outer_iterations_55_num_samples_10000_num_epochs_5_dataset_mnist_optim_adam_activation_relu_sampling_method_committee_aligner_var_RNN_28/population_iteration_54.pt"

best_model_index = 1 #needs to be manually inspected from log file

blackbox_dict = torch.load(blackbox_dict_path)
blackbox_original_params_dict = torch.load(blackbox_og_params_dict_path)
final_population_dict = torch.load(final_population_dict_path)

blackbox = var_RNN(28, [28]) #input_size, layer_configs
blackbox.load_state_dict(blackbox_dict)

subs = [var_RNN(28, [28]) for i in range(10)]
print(final_population_dict.keys())
final_population = Population(subs=subs)
final_population.load_state_dict(final_population_dict)
final_population.best = best_model_index
best_model = final_population.subs[best_model_index]

final_population.evaluate(blackbox, model_type='rnn')

odict_keys(['subs.0.layers.0.weight_ih_l0', 'subs.0.layers.0.weight_hh_l0', 'subs.0.layers.0.bias_ih_l0', 'subs.0.layers.0.bias_hh_l0', 'subs.0.layers.1.weight', 'subs.0.layers.1.bias', 'subs.1.layers.0.weight_ih_l0', 'subs.1.layers.0.weight_hh_l0', 'subs.1.layers.0.bias_ih_l0', 'subs.1.layers.0.bias_hh_l0', 'subs.1.layers.1.weight', 'subs.1.layers.1.bias', 'subs.2.layers.0.weight_ih_l0', 'subs.2.layers.0.weight_hh_l0', 'subs.2.layers.0.bias_ih_l0', 'subs.2.layers.0.bias_hh_l0', 'subs.2.layers.1.weight', 'subs.2.layers.1.bias', 'subs.3.layers.0.weight_ih_l0', 'subs.3.layers.0.weight_hh_l0', 'subs.3.layers.0.bias_ih_l0', 'subs.3.layers.0.bias_hh_l0', 'subs.3.layers.1.weight', 'subs.3.layers.1.bias', 'subs.4.layers.0.weight_ih_l0', 'subs.4.layers.0.weight_hh_l0', 'subs.4.layers.0.bias_ih_l0', 'subs.4.layers.0.bias_hh_l0', 'subs.4.layers.1.weight', 'subs.4.layers.1.bias', 'subs.5.layers.0.weight_ih_l0', 'subs.5.layers.0.weight_hh_l0', 'subs.5.layers.0.bias_ih_l0', 'subs.5.layers.0.bias_hh

model_type:  rnn


In [4]:
#transformer

blackbox_dict_path = "transformer/seed_0_transx64_outer_iterations_55_num_samples_10000_num_epochs_5_dataset_mnist_optim_adam_activation_relu_sampling_method_committee_aligner_test3/black_box.pt"
blackbox_og_params_dict_path = "transformer/seed_0_transx64_outer_iterations_55_num_samples_10000_num_epochs_5_dataset_mnist_optim_adam_activation_relu_sampling_method_committee_aligner_test3/original_params_black_box.pt"
final_population_dict_path = "transformer/seed_0_transx64_outer_iterations_55_num_samples_10000_num_epochs_5_dataset_mnist_optim_adam_activation_relu_sampling_method_committee_aligner_test3/population_iteration_54.pt"

best_model_index = 1

blackbox_dict = torch.load(blackbox_dict_path)
blackbox_original_params_dict = torch.load(blackbox_og_params_dict_path)
final_population_dict = torch.load(final_population_dict_path)

blackbox = base_TransformerEncoder(28, [64]) #d_model, layer_configs (size of linear layer)
blackbox.load_state_dict(blackbox_dict)
 
subs = [base_TransformerEncoder(28, [64]) for i in range(10)]
print(final_population_dict.keys())
final_population = Population(subs=subs)
final_population.load_state_dict(final_population_dict)
final_population.best = best_model_index
best_model = final_population.subs[best_model_index]

final_population.evaluate(blackbox, model_type='trans')

odict_keys(['subs.0.encoder_layer.self_attn.in_proj_weight', 'subs.0.encoder_layer.self_attn.in_proj_bias', 'subs.0.encoder_layer.self_attn.out_proj.weight', 'subs.0.encoder_layer.self_attn.out_proj.bias', 'subs.0.encoder_layer.linear1.weight', 'subs.0.encoder_layer.linear1.bias', 'subs.0.encoder_layer.linear2.weight', 'subs.0.encoder_layer.linear2.bias', 'subs.0.encoder_layer.norm1.weight', 'subs.0.encoder_layer.norm1.bias', 'subs.0.encoder_layer.norm2.weight', 'subs.0.encoder_layer.norm2.bias', 'subs.0.linear.weight', 'subs.0.linear.bias', 'subs.1.encoder_layer.self_attn.in_proj_weight', 'subs.1.encoder_layer.self_attn.in_proj_bias', 'subs.1.encoder_layer.self_attn.out_proj.weight', 'subs.1.encoder_layer.self_attn.out_proj.bias', 'subs.1.encoder_layer.linear1.weight', 'subs.1.encoder_layer.linear1.bias', 'subs.1.encoder_layer.linear2.weight', 'subs.1.encoder_layer.linear2.bias', 'subs.1.encoder_layer.norm1.weight', 'subs.1.encoder_layer.norm1.bias', 'subs.1.encoder_layer.norm2.weight

model_type:  trans


In [ ]:
import torch
import torch.nn as nn

def get_activations(net, x):
    """
    Given a var_RNN network and input x, performs a forward pass
    while recording the activations from each layer. For each RNN layer,
    we record x[:, -1, :] (i.e. the final time-step's output). Then,
    the final linear layer is applied on the final RNN output.
    
    Returns:
        A list of activations (tensors) for each layer in net.layers.
        For the i-th layer, the tensor shape is (num_samples, num_neurons_i).
    """
    activations = []
    batch_size = x.size(0)
    
    # Forward through each RNN layer (all but last layer are RNNs)
    for i in range(len(net.layers) - 1):
        rnn_layer = net.layers[i]
        # Use the configured hidden size for the i-th RNN layer.
        hidden_size = net.layer_configs[i]
        h0 = x.new_zeros(1, batch_size, hidden_size)
        x, _ = rnn_layer(x, h0)
        # Record the final time-step's activation (one vector per sample).
        act = x[:, -1, :]  
        activations.append(act)
    
    # Final linear layer: note that we pass in the last time-step of the RNN output.
    linear_layer = net.layers[-1]
    x_linear = linear_layer(x[:, -1, :])
    activations.append(x_linear)
    
    return activations

def compute_cosine_similarity_matrix(act_a, act_b):
    """
    Given two activation tensors act_a and act_b (shape: [batch, neurons]),
    compute the cosine similarity between each pair of neurons (one from act_a,
    one from act_b) computed over the batch.
    
    Returns:
        A matrix of shape (num_neurons, num_neurons) where the entry (i, j)
        is the cosine similarity between neuron i from act_a and neuron j from act_b.
    """
    # Transpose so that each row is a vector of activations across samples.
    a = act_a.transpose(0, 1)  # shape: (num_neurons_a, batch)
    b = act_b.transpose(0, 1)  # shape: (num_neurons_b, batch)
    
    # Normalize each neuron's activation vector.
    a_norm = a / a.norm(dim=1, keepdim=True)
    b_norm = b / b.norm(dim=1, keepdim=True)
    
    # Cosine similarity is the dot product of normalized vectors.
    cos_sim = torch.mm(a_norm, b_norm.transpose(0, 1))
    return cos_sim

def find_best_matches(cos_sim_matrix):
    """
    For a given cosine similarity matrix, find for each neuron (each row)
    in netA the index of the neuron in netB (column) with highest similarity.
    
    Returns:
        best_matches: Tensor of indices (one for each neuron in netA).
        best_values: Tensor of corresponding cosine similarity values.
    """
    best_matches = torch.argmax(cos_sim_matrix, dim=1)
    best_values = torch.max(cos_sim_matrix, dim=1).values
    return best_matches, best_values

def analyze_networks(netA, netB, num_samples=100, seq_length=5, input_dim=28):
    """
    Given two var_RNN networks (netA and netB) with the same architecture,
    this function:
    
      1. Generates random input data of shape (num_samples, seq_length, input_dim).
      2. Performs a forward pass through each network while recording the activations
         of each layer (using final time-step activations for RNNs and the output of the
         Linear layer).
      3. Computes cosine similarity between corresponding neurons in netA and netB 
         (for every layer).
      4. Prints out which neuron in netB best corresponds to each neuron in netA
         based on cosine similarity, for each layer.
    
    Assumption:
      - Both netA and netB are instances of var_RNN (or similarly structured models)
        that use a nn.ModuleList stored as .layers and a list of hidden sizes stored in
        .layer_configs.
    """
    # Generate random input samples.
    x = torch.randn(num_samples, seq_length, input_dim)
    
    # Get activations for each layer in both networks.
    acts_A = get_activations(netA, x)
    acts_B = get_activations(netB, x)
    
    num_layers = len(acts_A)
    
    # For each corresponding layer in netA and netB, compute and print neuron correspondences.
    for layer_idx in range(num_layers):
        act_A = acts_A[layer_idx]
        act_B = acts_B[layer_idx]
        cos_sim = compute_cosine_similarity_matrix(act_A, act_B)
        best_matches, best_values = find_best_matches(cos_sim)
        
        # Determine layer type for printing purposes.
        layer_type = "RNN" if layer_idx < num_layers - 1 else "Linear"
        print(f"\nLayer {layer_idx} ({layer_type} Layer) Neuron Correspondences:")
        for neuron_idx in range(best_matches.numel()):
            match_idx = best_matches[neuron_idx].item()
            sim_val = best_values[neuron_idx].item()
            print(f"  Neuron {neuron_idx} in netA corresponds to Neuron {match_idx} in netB (cosine similarity: {sim_val:.4f})")

# =============================================================================
# Example usage:
#
# Assuming your var_RNN is defined as:
#
#   class var_RNN(nn.Module):
#       def __init__(self, input_size, layer_configs, batch_first=True):
#           super(var_RNN, self).__init__()
#           self.layers = nn.ModuleList()
#           self.layer_configs = layer_configs
#           current_input_size = input_size
#
#           for hidden_size in layer_configs:
#               self.layers.append(nn.RNN(input_size=current_input_size, 
#                                           hidden_size=hidden_size, 
#                                           num_layers=1, 
#                                           batch_first=batch_first))
#               current_input_size = hidden_size
#
#           # Append linear layer.
#           self.layers.append(nn.Linear(current_input_size, 10))
#
#       def forward(self, x):
#           batch_size = x.size(0)
#           for i in range(len(self.layers)-1):
#               h0 = x.new_zeros(1, batch_size, self.layer_configs[i])
#               layer = self.layers[i]
#               x, _ = layer(x, h0)
#           x = self.layers[-1](x[:, -1, :])
#           return x
#
# Create two networks with the same architecture:
#
# netA = var_RNN(input_size=28, layer_configs=[28, 28])
# netB = var_RNN(input_size=28, layer_configs=[28, 28])
#
# Then run:
# analyze_networks(netA, netB)
# =============================================================================
